##Load Cleaned Dataset

In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/cleaned_traffic_events.csv")

print(df.shape)
df.head()

(8171, 25)


,event_type,latitude,longitude,endlatitude,endlongitude,address,end_address,event_cause,requires_road_closure,start_datetime,...,veh_no,corridor,priority,created_date,police_station,zone,junction,hour,day_of_week,month
0,unplanned,13.040004,77.518099,0.000000,0.000000,"Mumbai Bengaluru Highway, Jalahalli Cross Junc...",NaN,vehicle_breakdown,False,2024-03-07 17:01:48.111000+00:00,...,FKN00GL0000,Tumkur Road,High,2024-03-07 17:03:51.164032+00,Peenya,NaN,NaN,17,Thursday,March
1,unplanned,12.921876,77.645158,0.000000,0.000000,"19th Main Road, Heavie Halcyon, Agara, HSR Lay...",NaN,vehicle_breakdown,False,2024-01-30 04:07:24.173000+00:00,...,FKN00GL0001,ORR East 1,High,2024-01-30 04:08:22.954979+00,HSR Layout,NaN,NaN,4,Tuesday,January
2,unplanned,12.955622,77.585708,0.000000,0.000000,"Lalbagh Main Road, Dr Sri Shantaveera Swami Ci...",NaN,others,False,2023-11-11 06:18:03.343000+00:00,...,NaN,Non-corridor,Low,2023-11-11 06:20:00.989398+00,Wilson Garden,Central Zone 2,UrvashiJunction,6,Saturday,November
3,unplanned,13.006147,77.579435,13.006239,77.579516,"Sankey Road, Bashyam Circle, Sadashiva Nagar, ...","Sankey Road, Palace Orchard Upper, Sadashiva N...",tree_fall,True,2024-03-07 17:56:55.061000+00:00,...,NaN,Non-corridor,Low,2024-03-07 17:58:56.696892+00,Sadashivanagar,NaN,NaN,17,Thursday,March
4,unplanned,12.953980,77.585233,0.000000,0.000000,"Lalbagh Fort Road, Lalbagh Main Gate Junction,...",NaN,vehicle_breakdown,False,2024-01-30 04:56:32.348000+00:00,...,FKN00GL0002,Non-corridor,Low,2024-01-30 04:58:55.937662+00,Wilson Garden,NaN,LalbaghMainGateJunc,4,Tuesday,January


# Feature Engineering

## Objective

Transform cleaned event records into meaningful features that improve predictive performance and capture traffic operational patterns.

The engineered features will focus on:

- Temporal patterns
- Event characteristics
- Traffic behavior
- Operational relevance

These features will be used for priority prediction modeling.

In [9]:
df['start_datetime'] = pd.to_datetime(
    df['start_datetime'],
    format='mixed',
    errors='coerce'
)

df['created_date'] = pd.to_datetime(
    df['created_date'],
    format='mixed',
    errors='coerce'
)

df['modified_datetime'] = pd.to_datetime(
    df['modified_datetime'],
    format='mixed',
    errors='coerce'
)

In [10]:
df['hour'] = df['start_datetime'].dt.hour
df['day_of_week'] = df['start_datetime'].dt.day_name()
df['month'] = df['start_datetime'].dt.month_name()

In [12]:
df['is_weekend'] = df['day_of_week'].isin(
    ['Saturday', 'Sunday']
).astype(int)

In [13]:
def peak_hour(hour):
    if (7 <= hour <= 10) or (17 <= hour <= 21):
        return 1
    return 0

df['is_peak_hour'] = df['hour'].apply(peak_hour)

In [14]:
event_mapping = {

    'vehicle_breakdown': 'Incident',
    'accident': 'Incident',
    'congestion': 'Incident',

    'construction': 'Infrastructure',
    'pot_holes': 'Infrastructure',
    'road_conditions': 'Infrastructure',
    'Debris': 'Infrastructure',
    'debris': 'Infrastructure',

    'water_logging': 'Weather',
    'Fog / Low Visibility': 'Weather',

    'public_event': 'Public Event',
    'procession': 'Public Event',
    'protest': 'Public Event',
    'vip_movement': 'Public Event',

    'tree_fall': 'Natural',

    'others': 'Other',
    'test_demo': 'Other'
}

df['event_category'] = (
    df['event_cause']
    .map(event_mapping)
)

In [15]:
df['event_category'].value_counts()

,count
event_category,
Incident,5395
Infrastructure,1200
Other,641
Weather,460
Natural,284
Public Event,191


In [16]:
df.isnull().sum().sort_values(
    ascending=False
)

,0
end_datetime,7681
end_address,7484
junction,5662
zone,4728
veh_no,3287
veh_type,3286
description,1360
endlatitude,167
endlongitude,167
corridor,20


# Feature Engineering

## Objective

Transform raw event attributes into meaningful predictive features that can improve model performance and capture operational traffic patterns.

---

## Features Created

### Temporal Features

- Hour of Event
- Day of Week
- Month
- Weekend Indicator
- Peak Hour Indicator

### Event Features

- Event Category

Event causes were grouped into broader operational categories such as:

- Incident
- Infrastructure
- Weather
- Public Event
- Natural
- Other

---

## Expected Benefits

- Capture temporal traffic behavior.
- Reduce categorical sparsity.
- Improve model generalization.
- Enhance interpretability for traffic management decisions.

In [17]:
drop_cols = [
    'end_datetime',
    'end_address',
    'endlatitude',
    'endlongitude',
    'veh_no'
]

df.drop(columns=drop_cols, inplace=True)

print(df.shape)

(8171, 23)


In [18]:
df['veh_type'] = df['veh_type'].fillna('Unknown')

In [19]:
df['zone'] = df['zone'].fillna('Unknown')

In [20]:
df['junction'] = df['junction'].fillna('Unknown')

In [21]:
df['description'] = df['description'].fillna('No Description')

In [22]:
df['corridor'] = df['corridor'].fillna('Unknown')
df['address'] = df['address'].fillna('Unknown')

In [23]:
df.isnull().sum().sort_values(ascending=False)

,0
event_type,0
latitude,0
longitude,0
address,0
event_cause,0
requires_road_closure,0
start_datetime,0
authenticated,0
modified_datetime,0
description,0


# Missing Value Treatment

## Observation

Several features contained missing values, with some columns exhibiting substantial sparsity.

Key affected features included:

- Vehicle Type
- Zone
- Junction
- Description

Additionally, end-location and end-time attributes contained extremely high missing percentages and limited predictive value.

---

## Insight

- Removing all records with missing values would result in significant data loss.
- Missing categorical information may itself carry operational meaning.
- Highly sparse end-state attributes are unlikely to contribute meaningful predictive signal.

---

## Action

- Dropped highly sparse end-state columns.
- Replaced missing categorical values with `Unknown`.
- Replaced missing descriptions with `No Description`.
- Preserved all available event records while eliminating null values from the modeling dataset.

In [24]:
print(df.shape)

df.isnull().sum().sum()

(8171, 23)


np.int64(0)

In [26]:
print("Shape:", df.shape)

print("\nTotal Missing Values:")
print(df.isnull().sum().sum())

print("\nColumns:")
print(df.columns.tolist())

Shape: (8171, 23)

Total Missing Values:
0

Columns:
['event_type', 'latitude', 'longitude', 'address', 'event_cause', 'requires_road_closure', 'start_datetime', 'authenticated', 'modified_datetime', 'description', 'veh_type', 'corridor', 'priority', 'created_date', 'police_station', 'zone', 'junction', 'hour', 'day_of_week', 'month', 'is_weekend', 'is_peak_hour', 'event_category']


In [28]:
df.to_csv(
    "feature_engineered_events.csv",
    index=False
)

print("Feature engineered dataset saved successfully!")

Feature engineered dataset saved successfully!


# Feature Engineering Summary

## Features Created

### Temporal Features
- Hour
- Day of Week
- Month
- Weekend Indicator
- Peak Hour Indicator

### Event Features
- Event Category

### Missing Value Treatment
- Replaced missing categorical values with `Unknown`
- Removed highly sparse end-state attributes
- Eliminated all remaining null values

## Final Dataset

- Rows: 8,171
- Columns: 23
- Missing Values: 0

The resulting dataset is ready for model preparation and training.